# 🔄 Data Preprocessing Pipeline
## Climate + Sanitary Data Integration for Health DRC

This notebook handles:
- Loading climate and epidemiological data
- Missing value imputation
- Temporal aggregation
- Feature normalization
- Data quality checks

In [15]:
import os
import sys
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from datetime import datetime, timedelta
import warnings

# 🔧 CRITICAL FIX: Add /app to Python path BEFORE any Django imports
if '/app' not in sys.path:
    sys.path.insert(0, '/app')
os.chdir('/app')

warnings.filterwarnings('ignore')

# Django setup
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'core.settings')
import django
django.setup()

from analytics.models import HealthZone, ClinicalData, Province

print("✅ All imports successful")


✅ All imports successful


## 1. Load Configuration

In [16]:
# Load preprocessing config
CONFIG_PATH = Path("/app/config/preprocess.yaml")
MODEL_DIR = Path("/app/models")
DATA_DIR = Path("/app/data")

def load_config(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

config = load_config(CONFIG_PATH)
print(f"📄 Config loaded: {config.keys()}")

📄 Config loaded: dict_keys(['provinces', 'cities', 'diseases', 'temporal', 'spatial', 'paths', 'missing_values'])


## 2. Load Climate Data

In [ ]:
def load_climate_data(climate_dir=None):
    """Load climate data from processed CSV or raw directory"""
    from pathlib import Path
    import glob
    
    # Try preprocessed CSV first
    processed_file = Path("/app/data/processed/climate_data.csv")
    if processed_file.exists():
        try:
            df = pd.read_csv(processed_file)
            print(f"✓ Loaded climate data from processed: {len(df)} records")
            print(f"  Columns: {df.columns.tolist()}")
            return df
        except Exception as e:
            print(f"✗ Error loading processed climate: {e}")
    
    # Fallback to raw climate directory
    if climate_dir is None:
        climate_dir = Path("/app/data/climat")
    
    frames = []
    files = glob.glob(str(climate_dir / '*.csv'))
    
    for file in files:
        try:
            df = pd.read_csv(file)
            frames.append(df)
            print(f"✓ Loaded {Path(file).name}")
        except Exception as e:
            print(f"✗ Error loading {file}: {e}")
    
    if frames:
        climate_df = pd.concat(frames, ignore_index=True)
        print(f"\n📊 Climate data shape: {climate_df.shape}")
        return climate_df
    else:
        print("⚠️ No climate data found")
        return None

climate_df = load_climate_data(DATA_DIR / 'climat')

✓ Loaded rdc_climate_monthly.csv

📊 Climate data shape: (60, 4)
   Columns: ['date', 'temperature', 'precipitation', 'relative_humidity']


## 3. Load Epidemiological Data from Database

In [18]:
def load_epidemiological_data():
    """Load all clinical data from Django ORM"""
    from django.db.models import Q
    
    # Maladies épidémiques cibles
    epidemic_diseases = {
        'Paludisme', 'Choléra', 'VIH/SIDA', 'Tuberculose'
    }
    
    data = []
    for disease in epidemic_diseases:
        records = ClinicalData.objects.filter(
            disease_name=disease
        ).values('date', 'health_zone', 'cases', 'deaths')
        
        df = pd.DataFrame(records)
        df['disease'] = disease
        data.append(df)
        print(f"✓ {disease}: {len(df)} records")
    
    if data:
        epi_df = pd.concat(data, ignore_index=True)
        print(f"\n📊 Epidemiological data shape: {epi_df.shape}")
        return epi_df
    return None

epi_df = load_epidemiological_data()

FieldError: Cannot resolve keyword 'date' into field. Choices are: cases_count, created_at, deaths_count, disease_name, health_zone, health_zone_id, id, month

## 4. Handle Missing Values & Normalization

In [ ]:
from sklearn.preprocessing import RobustScaler

def preprocess_data(epi_df, climate_df):
    """Preprocess epidemiological data with climatology"""
    
    if epi_df is None or len(epi_df) == 0:
        print("⚠️ No epidemiological data to preprocess")
        return None, climate_df, None
    
    print("🔧 Handling missing values...")
    
    # Find the cases column (could be 'cases', 'cases_count', or other numeric columns)
    cases_col = None
    for col in ['cases', 'cases_count', 'value']:
        if col in epi_df.columns:
            cases_col = col
            break
    
    if not cases_col:
        # Use first numeric column
        numeric_cols = epi_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            cases_col = numeric_cols[0]
        else:
            print("⚠️ No numeric columns found for cases")
            return epi_df, climate_df, None
    
    print(f"   Using '{cases_col}' for cases data")
    
    # Find date column
    date_col = None
    for col in ['date', 'month', 'YEAR (DISPLAY)']:
        if col in epi_df.columns:
            date_col = col
            break
    
    if date_col:
        try:
            epi_df[date_col] = pd.to_datetime(epi_df[date_col], errors='coerce')
            epi_df = epi_df.sort_values(date_col)
            print(f"   Sorted by {date_col}")
        except:
            pass
    
    # Forward fill for time series if there's a disease column
    if 'disease' in epi_df.columns:
        epi_df[cases_col] = epi_df.groupby('disease')[cases_col].transform(
            lambda x: x.ffill().fillna(0)
        )
    else:
        epi_df[cases_col] = epi_df[cases_col].ffill().fillna(0)
    
    print(f"   Missing {cases_col}: {epi_df[cases_col].isna().sum()}")
    
    # Climate interpolation
    if climate_df is not None:
        for col in climate_df.select_dtypes(include=[np.number]).columns:
            climate_df[col] = climate_df[col].interpolate(method='linear')
        print(f"   Climate missing values after interpolation: {climate_df.isna().sum().sum()}")
    
    # Normalize numerical features
    print("\n📐 Normalizing features...")
    
    scaler_cases = RobustScaler()
    epi_df[f'{cases_col}_normalized'] = scaler_cases.fit_transform(
        epi_df[[cases_col]]
    )
    print(f"   ✓ Normalized {cases_col}")
    
    if climate_df is not None:
        numeric_cols = climate_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            climate_df[numeric_cols] = RobustScaler().fit_transform(
                climate_df[numeric_cols]
            )
            print(f"   ✓ Normalized {len(numeric_cols)} climate columns")
    
    print("✅ Preprocessing complete")
    return epi_df, climate_df, scaler_cases

epi_processed, climate_processed, scaler = preprocess_data(epi_df, climate_df)

## 5. Save Processed Data

In [ ]:
import pickle

# Save processed data
if epi_processed is not None:
    epi_processed.to_csv(DATA_DIR / 'processed' / 'epi_data.csv', index=False)
    print("✅ Epidemiological data saved")

if climate_processed is not None:
    climate_processed.to_csv(DATA_DIR / 'processed' / 'climate_data.csv', index=False)
    print("✅ Climate data saved")

# Save scalers for inference
with open(DATA_DIR / 'processed' / 'scalers.pkl', 'wb') as f:
    pickle.dump({'cases': scaler}, f)
    print("✅ Scalers saved")

print(f"\n📁 All data saved to {DATA_DIR / 'processed'}")

## 6. Data Quality Report

In [ ]:
print("="*60)
print("📋 DATA QUALITY REPORT")
print("="*60)

if epi_processed is not None and len(epi_processed) > 0:
    print(f"\nEPIDEMIOLOGICAL DATA:")
    print(f"  Total records: {len(epi_processed)}")
    
    # Find date column
    date_col = None
    for col in ['date', 'month', 'YEAR (DISPLAY)']:
        if col in epi_processed.columns:
            date_col = col
            break
    
    if date_col:
        print(f"  Date range: {epi_processed[date_col].min()} to {epi_processed[date_col].max()}")
    
    if 'disease' in epi_processed.columns:
        print(f"  Diseases: {epi_processed['disease'].unique()[:10].tolist()}...")
    
    # Find cases column
    cases_col = None
    for col in ['cases', 'cases_count', 'cases_normalized', 'value']:
        if col in epi_processed.columns:
            cases_col = col
            break
    
    if cases_col:
        print(f"\n{cases_col} statistics:")
        print(epi_processed[cases_col].describe())
    else:
        print(f"\nColumns available: {epi_processed.columns.tolist()[:10]}")
else:
    print("\n⚠️ No epidemiological data")

if climate_processed is not None and len(climate_processed) > 0:
    print(f"\n\nCLIMATE DATA:")
    print(f"  Total records: {len(climate_processed)}")
    print(f"  Columns: {climate_processed.columns.tolist()}")
    print(f"\nClimate statistics:")
    print(climate_processed.describe())
else:
    print("\n⚠️ No climate data")

print("\n" + "="*60)
print("✅ Preprocessing pipeline complete")
print("="*60)